# 09 - RAG debugging

This notebook compares retrieval behavior when changing `k`, `chunk_size`, and `chunk_overlap`. It should be used to inspect good and bad retrieval cases before trusting the final answer.

In [ ]:
from llmkit.config.settings import get_settings
from llmkit.rag.answer import answer_with_context
from llmkit.rag.loaders import load_markdown_documents
from llmkit.rag.splitters import split_documents
from llmkit.rag.vectorstores import build_chroma_index, retrieve_chunks

settings = get_settings()
KNOWLEDGE_BASE = settings.knowledge_base_dir
BASE_VECTORSTORE = settings.vectorstore_dir
QUESTIONS = [
    "Who founded Insurellm?",
    "Who signed the Metropolitan Life Group contract that totals $1,098,000?",
]

In [ ]:
documents = load_markdown_documents(KNOWLEDGE_BASE)
experiments = [
    {"name": "small_chunks", "chunk_size": 250, "chunk_overlap": 40, "k": 3},
    {"name": "base_chunks", "chunk_size": 800, "chunk_overlap": 120, "k": 4},
]

for experiment in experiments:
    chunks = split_documents(
        documents,
        chunk_size=experiment["chunk_size"],
        chunk_overlap=experiment["chunk_overlap"],
    )
    vectorstore_path = BASE_VECTORSTORE / experiment["name"]
    build_chroma_index(chunks, persist_directory=vectorstore_path, source_path=str(KNOWLEDGE_BASE))
    print(f"\n===== Experiment: {experiment['name']} =====")
    for question in QUESTIONS:
        print(f"\nQuestion: {question}")
        retrieved = retrieve_chunks(question, persist_directory=vectorstore_path, k=experiment["k"])
        for chunk in retrieved:
            print(chunk.metadata)
            print(chunk.page_content[:220])
            print("---")
        answer = answer_with_context(question, retrieved)
        print("Answer:", answer.answer)
